# 05B_ResNet18_CNN_Encoder.ipynb
Loads 4-channel satellite images, builds a modified ResNet-18 CNN encoder, and trains a classifier. The encoder can later be reused for multimodal fusion.

In [ ]:

!pip -q install rasterio torch torchvision scikit-learn

from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet18
from sklearn.preprocessing import LabelEncoder

ROOT='/content/drive/MyDrive/FloodProject'
SENT=os.path.join(ROOT,'Sentinel')
LAND=os.path.join(ROOT,'Landsat')

IMAGE_SIZE=224
BATCH_SIZE=8
EPOCHS=10
LR=1e-4

files=glob.glob(os.path.join(SENT,'*.tif'))+glob.glob(os.path.join(LAND,'*.tif'))

# Labels inferred from filename (replace later with your final labels)
labels=[]
for f in files:
    name=os.path.basename(f).upper()
    if "FLASH" in name:
        labels.append("FLASH")
    elif "HEAT" in name:
        labels.append("HEAT")
    else:
        labels.append("OTHER")

encoder=LabelEncoder()
y=encoder.fit_transform(labels)

class SatelliteDataset(Dataset):
    def __init__(self, files, labels):
        self.files=files
        self.labels=labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        with rasterio.open(self.files[idx]) as src:
            img=src.read().astype(np.float32)

        img=np.clip(img/10000.0,0,1)
        x=torch.from_numpy(img)
        x=F.interpolate(x.unsqueeze(0),size=(IMAGE_SIZE,IMAGE_SIZE),
                        mode='bilinear',align_corners=False).squeeze(0)

        return x, torch.tensor(self.labels[idx]).long()

dataset=SatelliteDataset(files,y)

train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size

train_ds,val_ds=random_split(dataset,[train_size,val_size],
generator=torch.Generator().manual_seed(42))

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE)

model=resnet18(weights=None)

old=model.conv1
model.conv1=nn.Conv2d(
    4,
    old.out_channels,
    kernel_size=old.kernel_size,
    stride=old.stride,
    padding=old.padding,
    bias=False
)

num_classes=len(encoder.classes_)
model.fc=nn.Linear(model.fc.in_features,num_classes)

device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device)

criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=LR)

best_acc=0

for epoch in range(EPOCHS):

    model.train()
    train_correct=0
    train_total=0
    train_loss=0

    for images,labels in train_loader:

        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()

        outputs=model(images)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        train_loss+=loss.item()

        pred=outputs.argmax(1)

        train_correct+=(pred==labels).sum().item()
        train_total+=labels.size(0)

    train_acc=100*train_correct/train_total

    model.eval()

    val_correct=0
    val_total=0

    with torch.no_grad():

        for images,labels in val_loader:

            images=images.to(device)
            labels=labels.to(device)

            outputs=model(images)

            pred=outputs.argmax(1)

            val_correct+=(pred==labels).sum().item()
            val_total+=labels.size(0)

    val_acc=100*val_correct/val_total

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Validation Accuracy : {val_acc:.2f}%")

    if val_acc>best_acc:
        best_acc=val_acc
        torch.save(model.state_dict(),"cnn_encoder_resnet18.pth")

print("Best Validation Accuracy:",best_acc)
print("Saved model: cnn_encoder_resnet18.pth")
